> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTLDII6k/tuPEYMpbOeTSCuxmoICBuA/view?utm_content=DAGzTLDII6k&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h705121cba8)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install openai==2.11.0 gradio==6.2.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1 numexpr==2.14.1 wikipedia==1.4.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. LangChain V1.0 ReAct Agent 搭建

## 2.1 简介

看完了如何通过基础的 python 代码手搓 ReAct Agent，下面我们来看看如何使用 LangChain 框架来复现该系统。

## 2.2 LLM 模型

由于在 LangGraph 中，需要大模型支持工具调用（Function Calling）。因为通义千问明确在 LangChain 文档中表示支持，因此后续我们将使用 Qwen 系列模型进行演示。

In [ ]:
from langchain_community.chat_models import ChatTongyi
import os

llm = ChatTongyi(
  api_key=os.environ.get("DASHSCOPE_API_KEY"), 
  model="qwen-max")

response = llm.invoke("你好，请介绍一下你自己")

print(response.content)

## 2.3 Memory 记忆

在新版本里，我们不再需要通过 RunnableWithMessageHistory 的方式进行记忆的保留，在 LangGraph 下我们使用 InMemorySaver() 的方式进行保留。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

## 2.4 System Prompt 系统提示词
在 LangChain V0.3 开始就重构了 Agent 层，从而让开发者只描述能力，而不是再造 prompt。

所以只需要输入模型、工具列表、记忆以及系统提示词（会在每一次传入给模型时才传入，不会放入记忆中）

In [ ]:
system_prompt = "You are a helpful assistant"

## 2.5 Tool 工具

### 2.5.1 内置工具
假如我们需要使用 LangChain 内置的工具，我们首先需要使用一个 load_tools 工具，然后在里面写入对应工具的名称：

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

假如我们想添加更多的工具，我们可以在后面不断添加（部分依赖语言模型来执行计算或生成代码需要传入一个 LLM 实例）：

In [ ]:
tools = load_tools(["arxiv","llm-math", "wikipedia"], llm=llm)

### 2.3.2 自定义工具

在 LangChain里，定义工具的主要方法为使用 @tool 装饰器。这种方式最为简洁，只需在普通函数上方加上 @tool 装饰器,并且在内部加上文档字符串作为工具的介绍以及参数的说明，就能自动将该函数注册为一个可供智能体调用的工具。

In [ ]:
from langchain.tools import tool

@tool
def calculate(what: str) -> str:
    """
    calculate:
    e.g. calculate: 4 * 7 / 3
    Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
    """
    return str(eval(what))

假如我们不想在函数内部写入文档字符串，我们也可以在 @tool 中添加一些参数，这里的 name 表示工具的名称（假如没有就默认使用函数名称）。description 代表的函数的介绍，也就是与前面文档字符串的作用一致。

In [ ]:
from langchain.tools import tool

@tool(name = "calculator", description="""
  calculate:
  e.g. calculate: 4 * 7 / 3
  Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
  """)
def calculate(what: str) -> str:
    return str(eval(what))

在 LangChain v1.0 版本中还新增了一个基于 Pydantic 的输入格式审查工具。

In [ ]:
from pydantic import BaseModel, Field

class CalcInput(BaseModel):
    """Input for math calculation"""
    what: str = Field(description="A mathematical expression, e.g., '4 * 7 / 3'")
    
from langchain.tools import tool

@tool(args_schema=CalcInput)
def calculate(what: str) -> str:
    """
    calculate:
    e.g. calculate: 4 * 7 / 3
    Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
    """
    return str(eval(what))

在定义好两个工具后，我们可以将两个工具组合起来形成工具列表，等待后续传给 ReAct Agent 进行使用：

In [ ]:
from langchain.tools import tool

@tool
def calculate(what: str) -> str:
  """
  calculate:
  e.g. calculate: 4 * 7 / 3
  Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
  """
  return str(eval(what))

@tool
def average_dog_weight(name: str) -> str:
  """
  average_dog_weight:
  e.g. average_dog_weight: Collie
  returns average weight of a dog when given the breed
  """
  name = name.lower()
  if "scottish terrier" in name:
    return "Scottish Terriers average 20 lbs"
  elif "border collie" in name:
    return "A Border Collie's average weight is 37 lbs"
  elif "toy poodle" in name:
    return "A Toy Poodle's average weight is 7 lbs"
  else:
    return "An average dog weighs 50 lbs"
  
tools = [calculate, average_dog_weight]

## 2.6 系统组装

在准备好了一些基础的组件以后，我们使用 create_agent 的方法将这些内容组合起来：

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=llm, 
           tools=tools, 
           system_prompt=system_prompt, 
           checkpointer=memory)

然后我们同样需要设置 thread_id 并将问题进行传入：

In [ ]:
result1 = agent.invoke({"messages": [{"role": "user", "content": "I have 2 dogs, a border collie and a scottish terrier. What is their combined weight? Could you double it?"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1)

假如只需要答案的话：

In [ ]:
print(result1["messages"][-1].content)

## 2.7 使用 ChatInterface 实现页面构建
假如想快速构建一个 Gradio 的对话页面来测试我们的 agent，可以使用以下方式来实现：

- 定义一个 agent :

In [ ]:
from langchain.agents import create_agent
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt, checkpointer=memory)

- 基于 agent 定义一个 agent_response 函数：

In [ ]:
def agent_response(content, history):
  result1 = agent.invoke({"messages": [{"role": "user", "content": content}]}, config={"configurable": {"thread_id": "user_1"}})
  return result1["messages"][-1].content

- 创建并发布 ChatInterface 页面：

In [ ]:
import gradio as gr
demo = gr.ChatInterface(fn=agent_response)
demo.launch()

这样就可以去测试回复的内容是否准确，但假如希望能够看到工具调用情况，需要使用流式输出。